# Øvelser: sentimentanalyse med asent

**Social Data Science 1: lektion 7**

Øvelserne bruger `asent` fra slides (*asent: sentimentanalyse*) på det datasæt, vi har
preprocesseret sammen, `ft_lovforslag.csv`. Slides viste, hvad `asent` kan, som en ordbog ikke kan.
Her undersøger vi, hvor godt den passer til Folketingets sprog:

1. Sætninger, der ligner Folketinget: høflighed, faste vendinger og nægtelser.
2. Tonen i debatterne, og om den stemmer med ordbogen fra slides.
3. Hvilke ord der driver tonen.
4. Tonen sætning for sætning, og en minimumsvalidering.

Under **Hjælp** står, hvor I har set de funktioner, I skal bruge:

- **Slides**: lektion 7-slides, med navnet på slidet.
- **Preprocess-notebooken**: `l7_preprocess_dkparl_trinvis`, med trinnets nummer.
- **spaCy-notebooken**: `l7_spacy_supplement`, med trinnets nummer.

---

## Opsætning

`asent` kører oven på spaCy og kan ikke køre i browseren. spaCy-modellen og `asent` installeres
én gang i terminalen, som i spaCy-notebooken, trin 1:

```
pip install spacy asent
python -m spacy download da_core_news_sm
```

Kør derefter cellen herunder. Ret stien, så den peger på jeres egen kopi af `ft_lovforslag.csv`.

In [ ]:
import pandas as pd
import numpy as np

sti = "/Users/jeppefl/Library/CloudStorage/OneDrive-AalborgUniversitet/01_work/01_undervisning/02_sds1/01_slides/lektion-7/resources/ft_lovforslag.csv"
korpus = pd.read_csv(sti)

print(korpus.shape)

spaCy er langsom på hele materialet (spaCy-notebooken, trin 3.2). Øvelse 2-4 bruger derfor det
samme udsnit på 100 debatter som spaCy-notebooken, trin 3.1.

---

## 1. asent på Folketingets sprog

Slides viste, at `asent` kan se negation, som vores egen ordbog ikke kunne (*Hvad den kan, som
vores egen ikke kunne*). Men slides testede sætninger, der ligner nyhedstekst. Her tester vi
sætninger, der ligner Folketinget: høflighedsformler, *godt* i faste vendinger og nægtelser
med dansk ordstilling.

**Opgave:**

1. Lav pipelinen fra slides: `da_core_news_sm` med `sentencizer` og `asent_da_v1`.
2. Kør sætningerne herunder igennem, og lav en tabel med sætningen, dens `compound` og de ord,
   der drev scoren.

```python
test = pd.Series([
    "Det er en god beslutning for vores frihed.",
    "Det er ikke en god beslutning.",
    "Det er godt, at ministeren er her.",
    "Godt nok er forslaget dyrt, men vi støtter det.",
    "Vi kan godt tænke os at se på det.",
    "Tak for ordet, og tak til ministeren for et godt samarbejde.",
    "Vi støtter ikke forslaget.",
    "Vi kan ikke støtte forslaget.",
    "Regeringen svigter de ældre.",
])
```

**Hjælp:**

- Slides, *asent: sentimentanalyse*: `nlp.add_pipe("sentencizer")`, `nlp.add_pipe("asent_da_v1")` og `doc._.polarity`.
- Slides, *Hvorfor fik den den score?*: `tok._.polarity.polarity` for hvert token.
- spaCy-notebooken, trin 2: en lille funktion, der bruges på hvert `Doc`. Her kan den returnere en liste med de ord, der har en polaritet.
- `.map(nlp)` sender hver sætning gennem pipelinen og giver en Series med ét `Doc` per sætning.

In [ ]:
# Din kode her

**Kig efter i viewer:** Åbn tabellen. Hvilke scorer ville I være uenige i?

**Til diskussion:**

- Sammenlign *Vi støtter ikke forslaget* og *Vi kan ikke støtte forslaget*. Hvad er forskellen på
  de to sætninger, og hvorfor giver `asent` dem modsat fortegn?
- Hvordan behandler `asent` *godt* i *godt nok* og *kan godt tænke os*? Er det bedre end
  ordbogen fra slides?
- Er der ord, der får en polaritet, I ikke havde forventet?

*Jeres svar:*

---

## 2. Tonen i debatterne, og stemmer den med ordbogen?

Slides kørte `asent` på debatterne og fandt, at Folketinget er høfligt (*Folketinget er høfligt*
og *Per politikområde*). Her gør vi det samme på et udsnit og tjekker, om `asent` og ordbogen fra
slides er enige om, hvilke debatter der er positive.

**Opgave:**

1. Tag et udsnit på 100 debatter med `random_state=1`, som i spaCy-notebooken, trin 3.1, og kør
   teksterne gennem `nlp`. Tag tid på det.
2. Gem `compound` for hver debat i kolonnen `sentiment`.
3. Beskriv fordelingen. Hvor stor en andel af debatterne er negative?
4. Lav en oversigt med antal debatter og gennemsnitlig `sentiment` per politikområde.
5. Beregn ordbogens score for de samme debatter med ordbogen fra slides (*Vi bygger én*), både
   med og uden *godt*. Gem dem som `score` og `uden_godt`.
6. Beregn rangkorrelationen (Spearman) mellem `sentiment`, `score` og `uden_godt`.

```python
ordbog = {"god": 2, "godt": 2, "bedre": 2, "bedst": 3, "stærk": 2,
          "sand": 2, "sandhed": 2, "frihed": 3, "tryghed": 2, "stolt": 2,
          "dårlig": -2, "værre": -2, "værst": -3, "katastrofe": -3,
          "løgn": -3, "svigt": -2, "trussel": -2, "farlig": -2,
          "krise": -2, "vold": -3, "frygt": -2, "svag": -2}
```

**Hjælp:**

- spaCy-notebooken, trin 3.1 og 3.2: `.sample(100, random_state=1)`, `nlp.pipe(..., batch_size=50)` og `time.perf_counter()`.
- Slides, *Kør den på korpusset* (asent): `[doc._.polarity.compound for doc in ...]`.
- Preprocess-notebooken, trin 5.2: `.agg(antal="size", ...)`.
- Slides, *Hvordan den scorer* og *Kør den på korpusset*: tokenisering, `.map(ordbog)` og `.groupby(level=0).sum()`. `.where(ord != "godt")` sætter *godt* til `NaN`, så det ikke tæller med.
- `.corr(method="spearman")` giver rangkorrelationen mellem kolonnerne i en tabel.

In [ ]:
# Din kode her

**Kig efter i viewer:** Sortér `udsnit` efter `sentiment`. Læs begyndelsen af den mest negative og den mest positive debat.

**Til diskussion:**

- Hvor mange debatter er der per område i udsnittet? Kan I sige noget om forskelle mellem områderne?
- Hvor enige er `asent` og ordbogen? Hvad betyder det, at to mål for *tone* kun hænger delvist sammen?
- Hvilket af de to mål ville I stole mest på? Hvorfor?

*Jeres svar:*

---

## 3. Hvilke ord driver tonen?

Med ordbogen fra slides var *godt* det ord, der fyldte mest i scoren (*Hvad fylder i scoren?*).
Hvilke ord driver `asent`? Det er samme spørgsmål, nu for en sprogmodel.

**Opgave:**

1. Skriv en funktion `polaritetstabel(doc)`, der laver en tabel med de ord i et `Doc`, der har en
   polaritet, og deres polaritet.
2. Brug den på alle debatterne i `udsnit`, og sæt tabellerne sammen.
3. Summér polariteten for hvert ord. Hvilke ord trækker mest op, og hvilke trækker mest ned?
4. Hvor stor en andel af den samlede positive polaritet kommer fra *tak* og *godt*?

**Hjælp:**

- spaCy-notebooken, trin 1 og 3.3: `tokentabel()` og `pd.concat(tabeller.to_dict(), names=[...]).reset_index()`.
- Slides, *Hvorfor fik den den score?*: `tok._.polarity.polarity`.
- Preprocess-notebooken, trin 5.2: `.agg(antal="size", sum="sum")`.

In [ ]:
# Din kode her

**Kig efter i viewer:** Åbn `bidrag`. Sortér efter `sum`. Læs de 20 øverste og de 20 nederste.

**Til diskussion:**

- Hvilke af de mest positive ord er vurderinger, og hvilke er høflighed eller procedure?
- Hvilke af de mest negative ord beskriver et problem, som debatten handler om, snarere end en
  negativ holdning?
- Er der ord, der får en polaritet, som ikke giver mening?

*Jeres svar:*

---

## Hvad har I fundet?

Udfyld tabellen, når I er færdige. Den er en huskeliste til metodeafsnittet.

| Øvelse | Metode | Hvad fandt I? | Hvad skal man være opmærksom på? |
|---|---|---|---|
| 1 | `asent` på testsætninger | | |
| 2 | tonen i debatterne, sammenlignet med ordbogen | | |
| 3 | ord, der driver tonen | | |
| 4 | høflighed og minimumsvalidering | | |